In [20]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import io
import re
import numpy as np
from gensim.models import KeyedVectors

## Загрузка полных векторов

In [22]:
def load_vectors(fname):
    """Загружает текстовые векторы fastText в память (слово -> вектор)."""
    fin = io.open(fname, 'r', encoding='utf-8', newline='\n', errors='ignore')
    n, d = map(int, fin.readline().split())
    data = {}
    for line in fin:
        tokens = line.rstrip().split(' ')
        # Преобразуем строки в float32 — экономим память
        data[tokens[0]] = np.asarray(tokens[1:], dtype='float32')
    return data

In [23]:
print("Загружаю русские эмбеддинги...")
ru_emb = load_vectors('resources/cc.ru.300.vec')
print(f"Русских слов: {len(ru_emb)}")

print("Загружаю польские эмбеддинги...")
pl_emb = load_vectors('resources/cc.pl.300.vec')
print(f"Польских слов: {len(pl_emb)}")

Загружаю русские эмбеддинги...
Русских слов: 2000000
Загружаю польские эмбеддинги...
Польских слов: 2000000


## Загрузка учебного словаря и построение матриц X, Y

In [24]:
def load_train_dict(path):
    pairs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 2:
                continue
            ru_word, pl_word = parts
            if ru_word in ru_emb and pl_word in pl_emb:
                pairs.append((ru_word, pl_word))
    return pairs

train_pairs = load_train_dict('resources/train_dict_ru_pl.txt')

In [25]:
X = np.array([ru_emb[ru] for ru, pl in train_pairs])
Y = np.array([pl_emb[pl] for ru, pl in train_pairs])
print(f"X.shape = {X.shape}, Y.shape = {Y.shape}")

X.shape = (20935, 300), Y.shape = (20935, 300)


## Обучение W через ортогональный Procrustes

In [26]:
def procrustes(X, Y):
    """
    Ортогональное решение Procrustes.
    Находит W, минимизирующую ||W X - Y||_F при W^T W = I.
    """
    # Сингулярное разложение Y^T X
    U, S, Vt = np.linalg.svd(Y.T @ X)
    W = U @ Vt
    return W

W = procrustes(X, Y)
print(f"W.shape = {W.shape}")

# Проверим ортогональность: W^T W ≈ I
print("Ортогональность W:", np.allclose(W.T @ W, np.eye(W.shape[0]), atol=1e-3))

W.shape = (300, 300)
Ортогональность W: True


In [27]:
def translate_word_nn(word, ru_emb, pl_emb, W, topn=5):
    if word not in ru_emb:
        return ["[OOV]"]
    
    x = ru_emb[word]
    transformed = W @ x
    transformed_norm = transformed / np.linalg.norm(transformed)
    
    # Перебираем все польские слова (для простоты; в реальности — FAISS)
    scores = {}
    for pl_word, pl_vec in pl_emb.items():
        pl_vec_norm = pl_vec / (np.linalg.norm(pl_vec) + 1e-10)
        scores[pl_word] = float(np.dot(transformed_norm, pl_vec_norm))
    
    sorted_words = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [w for w, s in sorted_words[:topn]]

# Пример
print(translate_word_nn("кот", ru_emb, pl_emb, W)[0])

kot


## Метрика CSLS (формула 6 из статьи)
CSLS борется с проблемой «хабов» — слов, которые становятся ближайшими соседями для многих.

$CSLS(W x_s, y_t) = 2 cos(W x_s, y_t) − r_T(W x_s) − r_S(y_t)$

In [28]:
def build_csls_cache(ru_emb, pl_emb, W, K=10, max_words=50000, chunk=1000):
    """
    Считает r_T и r_S порциями по chunk строк.
    Полные матрицы схожести (50000x50000 float64 ~ 20 ГБ) не создаются — только float32-порции.
    """
    ru_words = list(ru_emb.keys())[:max_words]
    pl_words = list(pl_emb.keys())[:max_words]

    X_sub = np.asarray([ru_emb[w] for w in ru_words], dtype=np.float32)
    Y_sub = np.asarray([pl_emb[w] for w in pl_words], dtype=np.float32)

    X_sub /= (np.linalg.norm(X_sub, axis=1, keepdims=True) + 1e-10)
    Y_sub /= (np.linalg.norm(Y_sub, axis=1, keepdims=True) + 1e-10)

    # Отображаем русские в польское пространство (в float32)
    W = np.asarray(W, dtype=np.float32)
    WX = X_sub @ W.T
    WX /= (np.linalg.norm(WX, axis=1, keepdims=True) + 1e-10)

    n_ru, n_pl = len(ru_words), len(pl_words)

    # r_T: для каждого русского — среднее сходство с K ближайшими польскими
    r_T = np.empty(n_ru, dtype=np.float32)
    for s in range(0, n_ru, chunk):
        e = min(s + chunk, n_ru)
        sims = WX[s:e] @ Y_sub.T                     # (chunk, n_pl), float32
        top = np.argpartition(sims, n_pl - K, axis=1)[:, -K:]
        r_T[s:e] = np.take_along_axis(sims, top, axis=1).mean(axis=1)

    # r_S: для каждого польского — среднее сходство с K ближайшими русскими
    r_S = np.empty(n_pl, dtype=np.float32)
    for s in range(0, n_pl, chunk):
        e = min(s + chunk, n_pl)
        sims = Y_sub[s:e] @ WX.T
        top = np.argpartition(sims, n_ru - K, axis=1)[:, -K:]
        r_S[s:e] = np.take_along_axis(sims, top, axis=1).mean(axis=1)

    return ru_words, pl_words, WX, Y_sub, r_T, r_S
    
# Переводл с CSLS
def translate_word_csls(word, ru_emb, pl_emb, W,
                        ru_words, pl_words, WX, Y_sub, r_T, r_S,
                        topn=5):
    if word not in ru_emb or word not in ru_words:
        return ["[OOV]"]
    
    i = ru_words.index(word)
    # Сходство с польскими словами
    sims = WX[i] @ Y_sub.T  # (len(pl_words),)
    # CSLS
    csls_scores = 2 * sims - r_T[i] - r_S
    # Топ-n
    idx = np.argsort(-csls_scores)[:topn]
    return [pl_words[j] for j in idx]

In [29]:
import re

def tokenize(text):
    # Простая токенизация: только буквы, цифры, дефис
    return re.findall(r'[а-яёa-z0-9\-]+', text.lower())

def translate_text(text, ru_emb, pl_emb, W,
                   ru_words, pl_words, WX, Y_sub, r_T, r_S,
                   topn=1):
    tokens = tokenize(text)
    translated = []
    for tok in tokens:
        if tok in ru_emb:
            pl_words_top = translate_word_csls(
                tok, ru_emb, pl_emb, W,
                ru_words, pl_words, WX, Y_sub, r_T, r_S, topn=topn
            )
            translated.append(pl_words_top[0] if pl_words_top else tok)
        else:
            # Если слова нет в эмбеддингах — оставляем как есть
            translated.append(tok)
    return " ".join(translated)

In [30]:
# 1. Загрузка
ru_emb = load_vectors('resources/cc.ru.300.vec')
pl_emb = load_vectors('resources/cc.pl.300.vec')

# 2. Словарь
train_pairs = load_train_dict('resources/train_dict_ru_pl.txt')
X = np.array([ru_emb[ru] for ru, pl in train_pairs])
Y = np.array([pl_emb[pl] for ru, pl in train_pairs])

# 3. Procrustes
W = procrustes(X, Y)
print("W обучена")

# 4. CSLS-кэш
print("CSLS кэш")
ru_words, pl_words, WX, Y_sub, r_T, r_S = build_csls_cache(
    ru_emb, pl_emb, W, K=10, max_words=50000
)
print("Кэш готов")

# 5. Перевод текста
text_ru = "Кот сидит на окне и смотрит на улицу."
text_pl = translate_text(
    text_ru, ru_emb, pl_emb, W,
    ru_words, pl_words, WX, Y_sub, r_T, r_S
)
print("RU:", text_ru)
print("PL:", text_pl)

W обучена
CSLS кэш
Кэш готов
RU: Кот сидит на окне и смотрит на улицу.
PL: kot siedzi na oknie i patrzy na ulicę


In [31]:
import os
import re

# ---------------------------------------------------------------
# Токенизация и перевод (используем функции из предыдущих блоков)
# ---------------------------------------------------------------

def tokenize(text):
    """Простая токенизация: только буквы, цифры, дефис. Приводит к нижнему регистру."""
    return re.findall(r'[а-яёa-z0-9\-]+', text.lower())

def translate_text(text, ru_emb, pl_emb, W,
                   ru_words, pl_words, WX, Y_sub, r_T, r_S,
                   topn=1, show_oov=True):
    """
    Пословный перевод текста с русского на польский.
    Возвращает кортеж: (перевод, список_OOV_слов).
    """
    tokens = tokenize(text)
    translated = []
    oov = []  # слова, которых нет в русских эмбеддингах

    for tok in tokens:
        if tok in ru_emb:
            pl_top = translate_word_csls(
                tok, ru_emb, pl_emb, W,
                ru_words, pl_words, WX, Y_sub, r_T, r_S, topn=topn
            )
            translated.append(pl_top[0] if pl_top else tok)
        else:
            translated.append(tok)   # оставляем как есть
            oov.append(tok)

    if show_oov and oov:
        print(f"  [!] Слова вне словаря ({len(oov)}): {', '.join(sorted(set(oov)))}")

    return " ".join(translated), oov


# ---------------------------------------------------------------
# 1. Перевод одного текстового файла
# ---------------------------------------------------------------

def translate_file(input_path, output_path=None, topn=1):
    """
    Читает txt-файл, переводит построчно, сохраняет результат.
    Если output_path=None — сохраняет рядом с исходным с суффиксом '_pl'.
    """
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    print(f"\n=== Перевод файла: {input_path} ===")
    translated_lines = []

    for i, line in enumerate(lines, 1):
        line = line.strip()
        if not line:
            translated_lines.append("")
            continue

        translation, _ = translate_text(
            line, ru_emb, pl_emb, W,
            ru_words, pl_words, WX, Y_sub, r_T, r_S,
            topn=topn
        )
        translated_lines.append(translation)
        # Печатаем прогресс каждые 10 строк, чтобы не засорять вывод
        if i % 10 == 0 or i == len(lines):
            print(f"  обработано строк: {i}/{len(lines)}")

    # Сохранение
    if output_path is None:
        base, ext = os.path.splitext(input_path)
        output_path = f"{base}_pl{ext}"

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(translated_lines))

    print(f"  Готово! Результат сохранён в: {output_path}")
    return output_path


# ---------------------------------------------------------------
# 2. Интерактивный перевод своих сообщений
# ---------------------------------------------------------------

def interactive_translator():
    """Цикл: пользователь вводит русский текст, получает польский перевод."""
    print("\n=== Интерактивный переводчик (RU → PL) ===")
    print("Введите текст на русском и нажмите Enter.")
    print("Для выхода введите: exit / quit / выход\n")

    while True:
        try:
            user_input = input("RU> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nВыход.")
            break

        if user_input.lower() in ('exit', 'quit', 'выход', ''):
            print("Выход.")
            break

        translation, _ = translate_text(
            user_input, ru_emb, pl_emb, W,
            ru_words, pl_words, WX, Y_sub, r_T, r_S,
            topn=1
        )
        print(f"PL> {translation}\n")

In [32]:
translate_file('examples/вирус.txt', output_path='examples/virus_polish.txt')


=== Перевод файла: examples/вирус.txt ===
  [!] Слова вне словаря (2): 35412, flemming
  [!] Слова вне словаря (3): илиаде, приама, эшнунна
  [!] Слова вне словаря (2): аристею, артемиде
  [!] Слова вне словаря (4): гидрофобию, плиний, средиземноморью, цельс
  [!] Слова вне словаря (2): антарктиду, безконтрольный
  [!] Слова вне словаря (1): пастер
  [!] Слова вне словаря (12): вонна, пастер, пастера, спиномозгового, спиномозговом, теодора, укусившая, францусзкой, эльзаса
  [!] Слова вне словаря (4): адельчи, пастеровского, пуркиньемозжечка, ремлинже
  [!] Слова вне словаря (6): пастеровском, пиван, порт-оф-спэйн, тобаго, тринидад
  обработано строк: 10/21
  [!] Слова вне словаря (6): абсорбирующимися, рабдовирусов, рнк-зависимой-рнк-полимеразы, рнк-содержащий, рнк-содержащих, транспортируемыми
  [!] Слова вне словаря (1): рнк-содержащих
  [!] Слова вне словаря (1): иннервированные
  [!] Слова вне словаря (1): гидрофобией
  [!] Слова вне словаря (1): пастера
  [!] Слова вне словаря (6

'examples/virus_polish.txt'

In [33]:
translate_file('examples/Маша и медведь.txt', output_path='examples/medved.txt')


=== Перевод файла: examples/Маша и медведь.txt ===
  [!] Слова вне словаря (1): машеньку
  [!] Слова вне словаря (1): видит-стоит
  обработано строк: 10/55
  [!] Слова вне словаря (1): машеньку
  [!] Слова вне словаря (1): потужила
  [!] Слова вне словаря (1): машеньке
  обработано строк: 20/55
  [!] Слова вне словаря (1): машеньке
  [!] Слова вне словаря (1): шёл-шёл
  обработано строк: 30/55
  [!] Слова вне словаря (2): шёл-шёл
  обработано строк: 40/55
  [!] Слова вне словаря (2): машеньки, отпирайте
  обработано строк: 50/55
  [!] Слова вне словаря (1): видят-
  [!] Слова вне словаря (2): живёхонька, здоровёхонька
  [!] Слова вне словаря (1): машеньку
  обработано строк: 55/55
  Готово! Результат сохранён в: examples/medved.txt


'examples/medved.txt'

In [34]:
my_text = "Когда-то тебе нужно сделать то За что тебя, мой друг, никто тут не похвалит"
result, _ = translate_text(
    my_text, ru_emb, pl_emb, W,
    ru_words, pl_words, WX, Y_sub, r_T, r_S
)
print("RU:", my_text)
print("PL:", result)

RU: Когда-то тебе нужно сделать то За что тебя, мой друг, никто тут не похвалит
PL: kiedyś tobie wystarczy zrobić to za jednak ciebie mój nawzajem nikt tutaj bo [OOV]


In [35]:
import os
import re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

# Отключаем лишние предупреждения
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

MODEL_NAME = "xlm-roberta-base"
print("Загружаю XLM-RoBERTa...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Модель загружена. Устройство: {device}")

Загружаю XLM-RoBERTa...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 904.17it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена. Устройство: cpu


In [36]:
def get_contextual_embeddings(sentence, tokenizer, model, device):
    """
    Пропускает предложение через XLM-R и возвращает:
    - список слов
    - список векторов (по одному на слово)
    """
    encoded = tokenizer(
        sentence,
        return_tensors="pt",
        padding=True,
        truncation=True,
        return_offsets_mapping=True
    )
    
    with torch.no_grad():
        outputs = model(
            input_ids=encoded["input_ids"].to(device),
            attention_mask=encoded["attention_mask"].to(device)
        )
    
    token_embeddings = outputs.last_hidden_state[0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    offsets = encoded["offset_mapping"][0].cpu().numpy()
    
    words = []
    word_vectors = []
    current_word = ""
    current_start = None
    current_vectors = []
    
    for i, (start, end) in enumerate(offsets):
        if start == 0 and end == 0:
            continue
        token_str = tokens[i]
        if token_str.startswith("▁") and current_word:
            words.append(current_word)
            word_vectors.append(np.mean(current_vectors, axis=0))
            current_word = ""
            current_start = None
            current_vectors = []
        if current_start is None:
            current_start = start
        clean_token = token_str.replace("▁", "")
        current_word += clean_token
        current_vectors.append(token_embeddings[i])
    
    if current_word:
        words.append(current_word)
        word_vectors.append(np.mean(current_vectors, axis=0))
    
    return words, np.array(word_vectors)

In [37]:
def get_word_vector_in_context(word, sentence, tokenizer, model, device):
    """Возвращает контекстуальный вектор для слова в предложении."""
    words, vectors = get_contextual_embeddings(sentence, tokenizer, model, device)
    word_lower = word.lower()
    for i, w in enumerate(words):
        if w.lower() == word_lower:
            return vectors[i]
    return None


def choose_best_translation(word_ru, sentence_ru, candidates_pl,
                            tokenizer, model, device,
                            ru_emb, pl_emb):
    """Выбирает лучший перевод среди кандидатов с учётом контекста."""
    if not candidates_pl:
        return None
    if len(candidates_pl) == 1:
        return candidates_pl[0]
    
    ctx_vec_ru = get_word_vector_in_context(
        word_ru, sentence_ru, tokenizer, model, device
    )
    if ctx_vec_ru is None:
        return candidates_pl[0]
    
    ctx_vec_ru_norm = ctx_vec_ru / (np.linalg.norm(ctx_vec_ru) + 1e-10)
    
    best_candidate = candidates_pl[0]
    best_score = -np.inf
    
    for cand in candidates_pl:
        cand_vec = get_word_vector_in_context(
            cand, cand, tokenizer, model, device
        )
        if cand_vec is None:
            continue
        cand_vec_norm = cand_vec / (np.linalg.norm(cand_vec) + 1e-10)
        score = float(np.dot(ctx_vec_ru_norm, cand_vec_norm))
        if score > best_score:
            best_score = score
            best_candidate = cand
    
    return best_candidate

In [38]:
def tokenize(text):
    return re.findall(r'[а-яёa-z0-9\-]+', text.lower())


def translate_text_contextual(text_ru,
                              ru_emb, pl_emb, W,
                              ru_words, pl_words, WX, Y_sub, r_T, r_S,
                              tokenizer, model, device,
                              topn=5, use_context=True):
    """Переводит русский текст на польский."""
    tokens = tokenize(text_ru)
    translated = []
    
    for tok in tokens:
        if tok not in ru_emb:
            translated.append(tok)
            continue
        
        candidates = translate_word_csls(
            tok, ru_emb, pl_emb, W,
            ru_words, pl_words, WX, Y_sub, r_T, r_S,
            topn=topn
        )
        
        if not candidates:
            translated.append(tok)
            continue
        
        if use_context and len(candidates) > 1:
            best = choose_best_translation(
                tok, text_ru, candidates,
                tokenizer, model, device,
                ru_emb, pl_emb
            )
            translated.append(best if best else candidates[0])
        else:
            translated.append(candidates[0])
    
    return " ".join(translated)

In [39]:
def translate_file_contextual(input_path, output_path=None,
                              ru_emb=None, pl_emb=None, W=None,
                              ru_words=None, pl_words=None,
                              WX=None, Y_sub=None, r_T=None, r_S=None,
                              tokenizer=None, model=None, device=None,
                              topn=5, use_context=True,
                              parallel=False):
    """Читает txt-файл, переводит построчно, сохраняет результат."""
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    
    print(f"\n=== Перевод файла: {input_path} ===")
    print(f"Всего строк: {len(lines)}")
    print(f"use_context = {use_context}")   # ← отладка
    
    translated_lines = []
    oov_all = set()
    
    for i, line in enumerate(lines, 1):
        if not line.strip():
            translated_lines.append("")
            continue
        
        translation = translate_text_contextual(
            line,
            ru_emb, pl_emb, W,
            ru_words, pl_words, WX, Y_sub, r_T, r_S,
            tokenizer, model, device,          # ← ЯВНО передаём
            topn=topn,                          # ← ЯВНО
            use_context=use_context             # ← ЯВНО
        )
        
        if parallel:
            translated_lines.append(f"RU: {line}\nPL: {translation}\n")
        else:
            translated_lines.append(translation)
        
        for tok in tokenize(line):
            if tok not in ru_emb:
                oov_all.add(tok)
        
        if i % 5 == 0 or i == len(lines):
            print(f"  обработано строк: {i}/{len(lines)}")
    
    if output_path is None:
        base, ext = os.path.splitext(input_path)
        suffix = "_pl_parallel" if parallel else "_pl"
        output_path = f"{base}{suffix}{ext}"
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(translated_lines))
    
    print(f"  Готово! Сохранено в: {output_path}")
    if oov_all:
        print(f"  Слов вне словаря: {len(oov_all)}")
    
    return output_path

In [40]:
FILE_1 = 'examples/вирус.txt'
FILE_2 = 'examples/Маша и медведь.txt'

translate_file_contextual(
    FILE_1,
    ru_emb=ru_emb, pl_emb=pl_emb, W=W,
    ru_words=ru_words, pl_words=pl_words,
    WX=WX, Y_sub=Y_sub, r_T=r_T, r_S=r_S,
    tokenizer=tokenizer, model=model, device=device,
    topn=5,
    use_context=False,   # ← отключаем XLM-R
    parallel=False
)

translate_file_contextual(
    FILE_2,
    ru_emb=ru_emb, pl_emb=pl_emb, W=W,
    ru_words=ru_words, pl_words=pl_words,
    WX=WX, Y_sub=Y_sub, r_T=r_T, r_S=r_S,
    tokenizer=tokenizer, model=model, device=device,
    topn=5, use_context=True, parallel=False
)


=== Перевод файла: examples/вирус.txt ===
Всего строк: 21
use_context = False
  обработано строк: 5/21
  обработано строк: 10/21
  обработано строк: 15/21
  обработано строк: 20/21
  обработано строк: 21/21
  Готово! Сохранено в: examples/вирус_pl.txt
  Слов вне словаря: 53

=== Перевод файла: examples/Маша и медведь.txt ===
Всего строк: 55
use_context = True
  обработано строк: 5/55
  обработано строк: 10/55
  обработано строк: 15/55
  обработано строк: 20/55
  обработано строк: 25/55
  обработано строк: 30/55
  обработано строк: 35/55
  обработано строк: 40/55
  обработано строк: 45/55
  обработано строк: 50/55
  обработано строк: 55/55
  Готово! Сохранено в: examples/Маша и медведь_pl.txt
  Слов вне словаря: 10


'examples/Маша и медведь_pl.txt'